In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

np.random.seed(42)
n_samples, n_features = 200, 10

X = np.random.randn(n_samples, n_features)

true_weights = np.array([5,-3,2,4,0,0,0,0,0,0])
y = X @ true_weights + np.random.randn(n_samples)

df = pd.DataFrame(X, columns = [f"feature_{i}" for i in range(n_features)])
df['target'] = y
print(df.head())

   feature_0  feature_1  feature_2  feature_3  feature_4  feature_5  \
0   0.496714  -0.138264   0.647689   1.523030  -0.234153  -0.234137   
1  -0.463418  -0.465730   0.241962  -1.913280  -1.724918  -0.562288   
2   1.465649  -0.225776   0.067528  -1.424748  -0.544383   0.110923   
3  -0.601707   1.852278  -0.013497  -1.057711   0.822545  -1.220844   
4   0.738467   0.171368  -0.115648  -0.301104  -1.478522  -0.719844   

   feature_6  feature_7  feature_8  feature_9     target  
0   1.579213   0.767435  -0.469474   0.542560   9.610682  
1  -1.012831   0.314247  -0.908024  -1.412304  -8.233614  
2  -1.150994   0.375698  -0.600639  -0.291694   1.649216  
3   0.208864  -1.959670  -1.328186   0.196861 -13.131167  
4  -0.460639   1.057122   0.343618  -1.763040  -0.151098  


In [ ]:
X = df.drop('target', axis = 1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
print(X_train.shape, X_test.shape)

(160, 10) (40, 10)


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train only
X_test_scaled = scaler.transform(X_test)          # just transform test — never fit on it

In [ ]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge': Ridge(alpha=1.0),
    'Lasso': Lasso(alpha=0.1),
    'Elastic Net': ElasticNet(alpha=0.1, l1_ratio=0.5)
}

results = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    results[name] = {
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R2': r2_score(y_test, y_pred)
    }

results_df = pd.DataFrame(results).T
print(results_df)

# .fit() learns the weights, .predict() uses them

                        MAE      RMSE        R2
Linear Regression  0.715657  0.882448  0.983570
Ridge              0.717377  0.889509  0.983306
Lasso              0.731221  0.905791  0.982689
Elastic Net        0.786612  0.999609  0.978917


In [ ]:
coef_df = pd.DataFrame({
    'feature': X.columns,
    'true_weight': true_weights,
    'Linear': models['Linear Regression'].coef_,
    'Ridge': models['Ridge'].coef_,
    'Lasso': models['Lasso'].coef_,
    'ElasticNet': models['Elastic Net'].coef_
})
print(coef_df.round(2))

     feature  true_weight  Linear  Ridge  Lasso  ElasticNet
0  feature_0            5    4.73   4.70   4.61        4.45
1  feature_1           -3   -3.23  -3.21  -3.14       -3.04
2  feature_2            2    1.87   1.87   1.83        1.81
3  feature_3            4    4.18   4.15   4.08        3.92
4  feature_4            0    0.06   0.05   0.00        0.00
5  feature_5            0   -0.08  -0.08  -0.00       -0.03
6  feature_6            0   -0.08  -0.07  -0.00       -0.00
7  feature_7            0   -0.06  -0.06  -0.00       -0.04
8  feature_8            0   -0.05  -0.05  -0.00       -0.00
9  feature_9            0    0.14   0.13   0.03        0.08


In [ ]:
from sklearn.linear_model import RidgeCV
import numpy as np

# list of alpha values to try
alphas_to_try = [0.001, 0.01, 0.1, 1, 10, 100]

ridge_cv = RidgeCV(alphas=alphas_to_try, cv=5)
ridge_cv.fit(X_train_scaled, y_train)

print("Best alpha found:", ridge_cv.alpha_)

Best alpha found: 1.0


In [ ]:
from sklearn.metrics import mean_squared_error

 #Use That Best Model Normally
y_pred = ridge_cv.predict(X_test_scaled)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print("Test RMSE with best alpha:", rmse)

Test RMSE with best alpha: 0.8895085262101379


In [ ]:
#Same Thing for Lasso and Elastic Net
from sklearn.linear_model import LassoCV, ElasticNetCV

lasso_cv = LassoCV(alphas=alphas_to_try, cv=5)
lasso_cv.fit(X_train_scaled, y_train)
print("Best Lasso alpha:", lasso_cv.alpha_)

elastic_cv = ElasticNetCV(alphas=alphas_to_try, l1_ratio=[0.2, 0.5, 0.8], cv=5)
elastic_cv.fit(X_train_scaled, y_train)
print("Best ElasticNet alpha:", elastic_cv.alpha_, "| Best l1_ratio:", elastic_cv.l1_ratio_)

Best Lasso alpha: 0.1
Best ElasticNet alpha: 0.01 | Best l1_ratio: 0.8


Hyperparameter tuning = trying several values, checking each one fairly using cross-validation (not the test set), and keeping whichever value scored best on average

In [ ]:
# grid search - try every combination
# Grid Search is exhaustive — tries every single combination in your grid.
#  With 6 alphas × 5 l1_ratios = 30 combinations × 5 folds = 150 model fits.
#   This gets expensive fast as you add more hyperparameters or values
#    (the "curse of dimensionality" strikes tuning too)

from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import ElasticNet

param_grid = {
    'alpha': [0.001, 0.01, 0.1, 1, 10, 100],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
}

grid_search = GridSearchCV(
    ElasticNet(max_iter=10000),
    param_grid,
    cv=5,
    scoring='neg_mean_squared_error',
    n_jobs=-1  # use all CPU cores
)
grid_search.fit(X_train_scaled, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV score (RMSE):", np.sqrt(-grid_search.best_score_))

best_model = grid_search.best_estimator_

Best params: {'alpha': 0.01, 'l1_ratio': 0.9}
Best CV score (RMSE): 1.042714506197071


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, loguniform

param_dist = {
    'alpha': loguniform(1e-3, 1e2),   # samples from log-scale range
    'l1_ratio': uniform(0, 1)
}

random_search = RandomizedSearchCV(
    ElasticNet(max_iter=10000),
    param_distributions=param_dist,
    n_iter=30,          # only try 30 random combinations, not all
    cv=5,
    scoring='neg_mean_squared_error',
    random_state=42,
    n_jobs=-1
)
random_search.fit(X_train_scaled, y_train)
print("Best params:", random_search.best_params_)

# Why Random Search often works better in practice, counter to intuition:
# if only 1-2 hyperparameters actually matter much (common in real models),
#  Grid Search wastes effort trying every combination of the unimportant ones,
#  while Random Search naturally covers more distinct values of each hyperparameter
#  within the same budget


Best params: {'alpha': np.float64(0.0745934328572655), 'l1_ratio': np.float64(0.9507143064099162)}


In [ ]:
from sklearn.linear_model import SGDRegressor

model = SGDRegressor(
    loss='squared_error',      # the loss function being minimized
    penalty='l2',               # 'l2' (Ridge), 'l1' (Lasso), 'elasticnet', or None
    alpha=0.01,                  # regularization strength
    l1_ratio=0.15,                # only used if penalty='elasticnet'
    learning_rate='constant',    # or 'optimal', 'invscaling', 'adaptive'
    eta0=0.01,                     # initial learning rate
    max_iter=1000,                # max epochs
    tol=1e-3,                      # stop early if improvement < tol
    early_stopping=True,           # use a validation split to decide when to stop
    random_state=42
)
model.fit(X_train_scaled, y_train)

SGDRegressor(alpha=0.01, early_stopping=True, learning_rate='constant',
             random_state=42)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, uniform

param_dist = {
    'alpha': loguniform(1e-5, 1e-1),
    'eta0': loguniform(1e-4, 1e-1),
    'l1_ratio': uniform(0, 1),
    'penalty': ['l2', 'l1', 'elasticnet']
}

search = RandomizedSearchCV(
    SGDRegressor(max_iter=2000, random_state=42),
    param_distributions=param_dist,
    n_iter=50,
    cv=5,
    scoring='neg_mean_squared_error',
    random_state=42
)
search.fit(X_train_scaled, y_train)
print(search.best_params_)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_stochastic_gradient.py:1608: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_stochastic_gradient.py:1608: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(


{'alpha': np.float64(0.029154431891537533), 'eta0': np.float64(0.006358358856676255), 'l1_ratio': np.float64(0.7080725777960455), 'penalty': 'l1'}


We use regression when we want to predict a continuous numeric value based on other known variables — like predicting a house's price from its size, forecasting someone's salary from years of experience, or estimating temperature from time of year. Instead of predicting a category or class, regression finds the mathematical relationship between input features and a numeric output, so once trained, the model can estimate that number for new, unseen data. It's useful anywhere the answer isn't "yes/no" or "which class," but "how much" or "how many."

# Regression — Classical ML Notes

---

## 1. Linear Regression

### Core Idea
Model target as a weighted linear combination of features.

$$\hat{y} = w_0 + w_1x_1 + w_2x_2 + \dots + w_nx_n$$

- $w_0$ = intercept (bias)
- $w_1...w_n$ = coefficients (how much $\hat y$ changes per unit change in that feature)

### Loss Function — Mean Squared Error
$$L(w) = \frac{1}{n}\sum_{i=1}^{n}(y_i - \hat{y}_i)^2$$

### How It's Solved
**a) Closed-form (Normal Equation)** — exact, one-shot solution:
$$w = (X^TX)^{-1}X^Ty$$
Fast for small/medium data; unstable/expensive for very large or highly correlated features.

**b) Gradient Descent** — iterative, needed for large data (see SGD section).

### Assumptions
1. Linearity — relationship between X and y is linear
2. Independence — observations don't affect each other
3. Homoscedasticity — residual variance is constant across X
4. No multicollinearity — features aren't highly correlated with each other
5. Normally distributed residuals

### Main Weakness
With many/correlated features → **overfitting**, unstable weights, poor generalization. This is why Ridge/Lasso/Elastic Net exist.

---

## 2. Ridge Regression (L2 Regularization)

Adds a penalty on squared weight magnitude:

$$L(w) = \sum_{i=1}^n(y_i - \hat{y}_i)^2 + \lambda\sum_{j=1}^n w_j^2$$

- $\lambda$ (`alpha`): 0 → plain linear regression; large → weights pushed toward 0 (underfitting)
- **Shrinks** weights toward zero but rarely makes them exactly zero
- Spreads weight across correlated features instead of letting one dominate

**Use when:** many features, especially correlated ones; you believe most/all features matter somewhat.

```python
from sklearn.linear_model import Ridge
model = Ridge(alpha=1.0)
model.fit(X_train, y_train)
```

---

## 3. Lasso Regression (L1 Regularization)

Penalizes absolute weight value:

$$L(w) = \sum_{i=1}^n(y_i - \hat{y}_i)^2 + \lambda\sum_{j=1}^n |w_j|$$

- Can shrink weights **exactly to zero** → built-in feature selection
- Geometric reason: L1 constraint region is a diamond (sharp corners on axes) → loss contours more likely to touch a corner (weight = 0). L2 constraint region is a smooth circle → no corners, so weights shrink but rarely hit exactly zero.

**Use when:** you suspect many features are irrelevant; want a sparse, interpretable model.

**Downside:** with a group of correlated features, Lasso arbitrarily picks one and zeroes out the rest.

```python
from sklearn.linear_model import Lasso
model = Lasso(alpha=0.1)
model.fit(X_train, y_train)
```

---

## 4. Elastic Net (L1 + L2 Combined)

$$L(w) = \sum_{i=1}^n(y_i - \hat{y}_i)^2 + \lambda_1\sum|w_j| + \lambda_2\sum w_j^2$$

sklearn parameterization:
$$\lambda \left[ \rho \sum|w_j| + \frac{(1-\rho)}{2}\sum w_j^2 \right]$$
- `l1_ratio` ($\rho$): 1 → pure Lasso, 0 → pure Ridge, 0.5 → balanced mix

**Use when:** high-dimensional data with groups of correlated features, want feature elimination *and* stability.

```python
from sklearn.linear_model import ElasticNet
model = ElasticNet(alpha=0.1, l1_ratio=0.5)
model.fit(X_train, y_train)
```

---

## 5. Summary Comparison

| Aspect | Linear Regression | Ridge (L2) | Lasso (L1) | Elastic Net |
|---|---|---|---|---|
| Penalty | None | $\sum w_j^2$ | $\sum \lvert w_j \rvert$ | Both |
| Shrinks weights? | No | Yes | Yes | Yes |
| Zeros out weights? | No | No | Yes | Yes |
| Feature selection? | No | No | Yes | Yes |
| Correlated features | Poorly | Well | Poorly (picks one) | Well |
| Best use case | Clean, few features | Many correlated features | Many irrelevant features | Both problems at once |

**Important:** Always **scale/standardize features** before Ridge/Lasso/Elastic Net — penalty is applied directly to weight magnitude, so unscaled features get penalized unevenly.

---

## 6. Evaluation Metrics for Regression

### MAE (Mean Absolute Error)
$$MAE = \frac{1}{n}\sum|y_i - \hat{y}_i|$$
Same unit as target, robust to outliers.

### MSE (Mean Squared Error)
$$MSE = \frac{1}{n}\sum(y_i - \hat{y}_i)^2$$
Penalizes large errors heavily; unit² (less interpretable); sensitive to outliers.

### RMSE (Root Mean Squared Error)
$$RMSE = \sqrt{MSE}$$
Same unit as target; still penalizes large errors more than MAE. Most commonly reported.

> Rule of thumb: RMSE ≥ MAE always. If RMSE ≫ MAE → a few large errors (outliers) exist.

### R² (Coefficient of Determination)
$$R^2 = 1 - \frac{\sum(y_i-\hat{y}_i)^2}{\sum(y_i-\bar{y})^2}$$
- R²=1 → perfect model, R²=0 → same as predicting the mean, R²<0 → worse than predicting the mean
- ⚠️ R² increases as you add features (even useless ones) → use **Adjusted R²** to penalize this.

---

## 7. The Full Training Workflow

```
1. Load data
2. Train/test split (BEFORE any preprocessing!)
3. Scale features — fit scaler on train only, transform both train & test
4. Train baseline models with default alpha
5. Tune alpha (and l1_ratio for Elastic Net) via cross-validation
6. Evaluate tuned models on the held-out test set
7. Inspect coefficients for interpretability / feature selection
```

### Code Skeleton

```python
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Scale (fit on train ONLY)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Train
model = Ridge(alpha=1.0)
model.fit(X_train_scaled, y_train)

# 4. Evaluate
y_pred = model.predict(X_test_scaled)
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2:", r2_score(y_test, y_pred))
```

### Common Pitfalls
- **Data leakage** — fitting scaler/imputer on full dataset before splitting. Always split first.
- **Evaluating on training data** — always looks better than it should; trust test/validation only.
- **Not scaling before Ridge/Lasso** — penalty term unfairly punishes/ignores unscaled features.
- **Picking alpha by eye** — use cross-validation instead of guessing.
- **Forgetting `max_iter`** — Lasso/ElasticNet use coordinate descent and may need more than default 1000 iterations to converge on real data.

---

## 8. Hyperparameter Tuning

### Parameters vs Hyperparameters

| | Parameters | Hyperparameters |
|---|---|---|
| What | Learned from data | Set before training, by you |
| Example | Weights $w$, bias $b$ | `alpha`, `l1_ratio`, learning rate, `max_iter`, `k` (KNN) |
| Who decides | The algorithm (`.fit()`) | You |

⚠️ Never tune hyperparameters by checking performance on the **test set** — that leaks test info into model selection. Use cross-validation on the training set instead.

### K-Fold Cross-Validation
Split training data into K folds (commonly 5 or 10). For each hyperparameter value:
- Train on K−1 folds, validate on the remaining fold
- Rotate through all K combinations
- Average the K validation scores → this is that hyperparameter's CV score

Pick whichever value has the best average CV score.

```python
from sklearn.model_selection import cross_val_score

scores = cross_val_score(Ridge(alpha=1.0), X_train_scaled, y_train,
                          cv=5, scoring='neg_mean_squared_error')
print("Mean CV RMSE:", np.sqrt(-scores.mean()))
```
(sklearn convention: scoring is "higher is better," so error metrics are negated — flip sign back.)

### Easiest Method — Built-in CV Classes
```python
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV

alphas_to_try = [0.001, 0.01, 0.1, 1, 10, 100]

ridge_cv = RidgeCV(alphas=alphas_to_try, cv=5)
ridge_cv.fit(X_train_scaled, y_train)
print("Best alpha:", ridge_cv.alpha_)

lasso_cv = LassoCV(alphas=alphas_to_try, cv=5, max_iter=10000)
lasso_cv.fit(X_train_scaled, y_train)

elastic_cv = ElasticNetCV(alphas=alphas_to_try, l1_ratio=[0.2, 0.5, 0.8], cv=5, max_iter=10000)
elastic_cv.fit(X_train_scaled, y_train)
print("Best alpha:", elastic_cv.alpha_, "| Best l1_ratio:", elastic_cv.l1_ratio_)
```
These objects are already trained with the best hyperparameter — just call `.predict()` directly.

### Grid Search vs Random Search (general purpose, more hyperparameters)

**Grid Search** — tries every combination exhaustively:
```python
from sklearn.model_selection import GridSearchCV

param_grid = {'alpha': [0.001, 0.01, 0.1, 1, 10, 100], 'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]}
grid_search = GridSearchCV(ElasticNet(max_iter=10000), param_grid, cv=5,
                            scoring='neg_mean_squared_error', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)
print(grid_search.best_params_)
```
Expensive: grows exponentially with number of hyperparameters/values.

**Random Search** — samples random combinations:
```python
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform, uniform

param_dist = {'alpha': loguniform(1e-3, 1e2), 'l1_ratio': uniform(0, 1)}
random_search = RandomizedSearchCV(ElasticNet(max_iter=10000), param_dist,
                                    n_iter=30, cv=5, scoring='neg_mean_squared_error',
                                    random_state=42, n_jobs=-1)
random_search.fit(X_train_scaled, y_train)
```
Often better in practice when only a few hyperparameters actually matter — covers more distinct values per hyperparameter within the same budget.

### Nested Cross-Validation
For rigorous final performance reporting (worth knowing for research writeups):
```
Outer loop: split into train/test (or outer K-folds)
  Inner loop: on outer-train, do CV to pick best hyperparameters
  Evaluate tuned model on outer-test (never seen during tuning)
```
Prevents overfitting to validation folds when hyperparameters were chosen using those same folds.

### One-Sentence Summary
**Hyperparameter tuning = trying several values, checking each fairly using cross-validation (never the test set), and keeping whichever value scored best on average.**

---

## 9. Solvers: Closed-form vs Coordinate Descent vs SGD

| Model | Solver | "Epochs"? |
|---|---|---|
| `LinearRegression` | Closed-form (Normal Equation) | No — solved in one step |
| `Ridge` | Closed-form | No |
| `Lasso` | Coordinate Descent | `max_iter` = iterations, not true epochs |
| `ElasticNet` | Coordinate Descent | Same as Lasso |
| `SGDRegressor` | True Stochastic Gradient Descent | Yes — real epochs + learning rate |

### Why No Epochs for Ridge/Linear Regression
Closed-form solution: $w = (X^TX + \lambda I)^{-1}X^Ty$ — pure linear algebra, computed in one shot, no iteration/loop.

### Why Lasso/ElasticNet Use Coordinate Descent (not gradient descent)
L1 penalty ($|w|$) isn't smooth/differentiable at zero, so the closed-form trick doesn't work. Instead:
- Optimize one weight at a time, holding all others fixed
- Cycle through all weights repeatedly until convergence
- `max_iter` = how many passes to allow (similar in spirit to epochs, but not the same mechanism as gradient descent)

---

## 10. SGD (Stochastic Gradient Descent)

`SGDRegressor` is **not a new model** — it's the same Linear/Ridge/Lasso/ElasticNet math, just solved with gradient descent instead of closed-form/coordinate descent.

```
                    WHAT you're modeling         HOW you solve for weights
Linear Regression → y = w·x + b            →  Closed-form  OR  SGD
Ridge (L2)        → y = w·x + b + L2 loss   →  Closed-form  OR  SGD
Lasso (L1)        → y = w·x + b + L1 loss   →  Coordinate Descent  OR  SGD
Elastic Net       → y = w·x + b + L1+L2     →  Coordinate Descent  OR  SGD
```

### Three Flavors of Gradient Descent

**1. Batch GD** — uses the entire training set per update. One update per epoch. Accurate direction, slow/memory-heavy on large data.

**2. Stochastic GD (true SGD)** — uses one random sample per update. Many updates per epoch. Noisy but can escape shallow local minima.

**3. Mini-Batch GD** — uses a small batch (e.g. 32, 64) per update. The practical default in sklearn/PyTorch/TensorFlow. Balances stability and speed.

### The Training Loop
```
For each epoch (full pass over training data):
    Shuffle the data
    For each mini-batch:
        1. Predict:    ŷ = w·x + b
        2. Loss:       L = (y - ŷ)²  [+ regularization]
        3. Gradient:   ∂L/∂w
        4. Update:     w := w - learning_rate × ∂L/∂w
    (optional) Check validation loss → early stopping decision
```

### Code — SGDRegressor Can Mimic Any of the 4 Models

```python
from sklearn.linear_model import SGDRegressor

# Like Linear Regression
sgd_linear = SGDRegressor(penalty=None, max_iter=1000, eta0=0.01, learning_rate='constant')

# Like Ridge
sgd_ridge = SGDRegressor(penalty='l2', alpha=0.01, max_iter=1000)

# Like Lasso
sgd_lasso = SGDRegressor(penalty='l1', alpha=0.01, max_iter=1000)

# Like Elastic Net
sgd_elastic = SGDRegressor(penalty='elasticnet', alpha=0.01, l1_ratio=0.5, max_iter=1000)
```

### Key Hyperparameters
- `loss` — loss function (e.g. `'squared_error'`)
- `penalty` — `'l2'`, `'l1'`, `'elasticnet'`, or `None`
- `alpha` — regularization strength
- `l1_ratio` — only used if `penalty='elasticnet'`
- `learning_rate` — schedule type:
  - `'constant'` — never changes
  - `'optimal'` — sklearn's heuristic decay
  - `'invscaling'` — decays as $\eta_0 / t^{power\_t}$
  - `'adaptive'` — stays constant while loss improves, halves when progress stalls
- `eta0` — initial learning rate
- `max_iter` — max epochs
- `tol` — stop early if improvement < tol
- `early_stopping` — use a validation split to decide when to stop

⚠️ **Feature scaling is critical for SGD** — unscaled features can make gradient descent diverge or converge painfully slowly (a learning rate tuned for a small-range feature will overshoot for a large-range feature).

### Solver Comparison Recap

| | Closed-form (`Ridge`) | Coordinate Descent (`Lasso`) | SGD (`SGDRegressor`) |
|---|---|---|---|
| Iterative? | No | Yes (not gradient-based) | Yes, true gradient descent |
| Needs learning rate? | No | No | Yes |
| Needs epochs/`max_iter`? | No | Yes (convergence check) | Yes (true epochs) |
| Scales to huge/streaming data? | Poorly | Moderately | Well |
| Precision on small data | Exact | Near-exact | Approximate |

### When to Actually Use SGDRegressor
- Dataset too large to fit in memory (need mini-batches)
- Online/streaming learning (model updates continuously as new data arrives)
- For most tabular datasets (thousands–low millions of rows): just use `Ridge`/`Lasso`/`ElasticNet` directly — faster, more precise, no learning rate to tune.

---

## 11. Known Gaps to Patch (Next Study Session)

These weren't covered yet but are commonly asked about in interviews:

1. **Polynomial Regression** — fitting nonlinear relationships by transforming features ($x, x^2, x^3, ...$) then still applying linear regression.
2. **Multicollinearity Diagnostics (VIF)** — Variance Inflation Factor, how to *detect* multicollinearity, not just fix it with Ridge.
3. **Residual Analysis** — plotting residuals to check linear regression assumptions (homoscedasticity, normality of errors).
4. **Bias-Variance Tradeoff** — the explicit framework tying regularization strength to underfitting/overfitting. One of the most commonly asked ML interview questions.

---



# Random Forest for Regression


---

## Core Idea

Mechanically identical to Random Forest for classification — same **bagging idea**, same **bootstrap sampling**, and same **random feature selection**.

The main differences are:

- Final prediction = **average** of all trees' outputs, rather than majority vote
- Use `RandomForestRegressor` instead of `RandomForestClassifier`
- Evaluate using regression metrics such as **MAE, RMSE, and R²** instead of Accuracy, Precision, Recall, and F1

---

## Assumed Dataset — Predicting House Price

We want to predict **house price (in lakhs NPR)** using:

- `area_sqft` — area of the house in square feet
- `house_age` — age of the house in years

Example pattern:

```text
Bigger area + newer house → generally higher price

In [1]:
import pandas as pd
import numpy as np

data = {
    'area_sqft':    [800, 1000, 1200, 1500, 1800, 2000, 2200, 2500, 2800, 3000, 3200, 3500],
    'house_age':    [20, 15, 18, 10, 8, 5, 12, 3, 6, 2, 4, 1],
    'price_lakhs':  [45, 55, 60, 78, 92, 105, 98, 130, 135, 152, 148, 165]
}

df = pd.DataFrame(data)
print(df)

    area_sqft  house_age  price_lakhs
0         800         20           45
1        1000         15           55
2        1200         18           60
3        1500         10           78
4        1800          8           92
5        2000          5          105
6        2200         12           98
7        2500          3          130
8        2800          6          135
9        3000          2          152
10       3200          4          148
11       3500          1          165


In [3]:
X = df[['area_sqft', 'house_age']]
y = df['price_lakhs']
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [4]:
from sklearn.ensemble import RandomForestRegressor

# No scaling needed — same as all tree-based models
model = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)

RandomForestRegressor(max_depth=5, random_state=42)

In [5]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R2:", r2_score(y_test, y_pred))

MAE: 10.303333333333327
RMSE: 11.380092266761283
R2: 0.9471976759989128


In [6]:
# Predict price for a 1600 sqft house that is 7 years old
new_house = [[1600, 7]]
predicted_price = model.predict(new_house)
print(f"Predicted price: {predicted_price[0]:.2f} lakhs")

# Try a few
test_houses = [[1000, 15], [2000, 5], [3000, 2], [1500, 20]]
for house in test_houses:
    pred = model.predict([house])[0]
    print(f"Area={house[0]} sqft, Age={house[1]} yrs → Predicted Price: {pred:.2f} lakhs")

Predicted price: 90.39 lakhs
Area=1000 sqft, Age=15 yrs → Predicted Price: 61.87 lakhs
Area=2000 sqft, Age=5 yrs → Predicted Price: 107.49 lakhs
Area=3000 sqft, Age=2 yrs → Predicted Price: 142.05 lakhs
Area=1500 sqft, Age=20 yrs → Predicted Price: 69.68 lakhs


/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [7]:
importances = model.feature_importances_
for feature, importance in zip(X.columns, importances):
    print(f"{feature}: {importance:.3f}")

area_sqft: 0.641
house_age: 0.359


In [8]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 10, None],
    'min_samples_leaf': [1, 2, 4]
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42), param_grid,
    cv=3,   # small dataset, keep folds low
    scoring='neg_mean_squared_error'
)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)
print("Best CV RMSE:", np.sqrt(-grid_search.best_score_))

Best params: {'max_depth': 5, 'min_samples_leaf': 1, 'n_estimators': 50}
Best CV RMSE: 16.066013817994804
